# A/B/C Ultrasound Scanner — GUI Documentation

This document describes the software developed by the UCL Ultrasonics Group for ultrasound scanning. The software coordinates a three-axis positioning rig, a signal generator, and an oscilloscope for ultrasound pulse-echo or -transmission acquisition and acoustic pressure-field measurements.

The GUI provides configuration, manual motion, excitation control, single-point A-mode acquisition, linear B-mode acquisition, and planar 3D acquisition with C-mode or pressure-field post-processing. The main files for the GUI application are `api/gui_main.py` and `api/src/pyside_gui.py`. 

**Release:** Unreleased (the GitHub repository currently has no published releases or tags)  
**Version:** `main@8b089f2`  
**Main-branch commit:** [`8b089f28e747a760e27b85275c479c162de33d80`](https://github.com/USonixGroup/A-B-C_scanner/commit/8b089f28e747a760e27b85275c479c162de33d80), dated 5 October 2025  

These identifiers describe the published GitHub `main` branch. A development branch or local working copy may contain newer changes.

> **Hardware safety:** The application can move motorised axes and enable an electrical output. Verify travel limits, axis directions, clearances, instrument addresses, excitation amplitude, and pulse timing before a real scan. Use **Dry Run** when validating a workflow.

## Launching and recommended workflow

Start the application from the repository root:

`
python api/gui_main.py
`

The required sequence:

1. Configure and test all connections in **Config**.
2. Establish a safe starting position in **Move Rig**.
3. Define the waveform in **Excitation Mode** and preview it.
4. Configure **A-Mode**, **B-Mode**, or **3D-Mode**.
5. Run once with **Dry Run** enabled.
6. Recheck the physical setup, disable Dry Run, and start hardware acquisition.
7. Inspect the live plot and log, then export the required data.

Settings are saved automatically to `api/settings/gui_settings.json`. This file can contain local IP addresses and instrument identifiers.

## 1. Config tab

The Config tab defines the three hardware connections and provides a non-motion connection test.

| Group | Parameter | Meaning | Example/default |
| --- | --- | --- | --- |
| Signal Generator | Name | PyMeasure instrument model/driver | `Agilent33500B` |
| Signal Generator | VISA Address | USB or network VISA resource | `USB0::...::INSTR` |
| Rig | Host | TCP/IP address of the motion controller | `192.168.1.250` |
| Rig | Port | TCP port exposed by the controller | `5001` |
| Oscilloscope | Name | Descriptive instrument name | `Lecroy` |
| Oscilloscope | VISA Address | VISA resource used for waveform capture | `TCPIP0::...::INSTR` |
| Oscilloscope | Sampling Rate | Acquisition rate, displayed in kHz | Hardware-dependent - default is 10 MHz |

The connection test attempts to reach the rig and open/query the instruments. It does not intentionally command an axis move. A successful network connection does not confirm that the rig workspace is clear or that scan directions are safe.

## 2. Move Rig tab

Use this tab for setup and positioning. The controller communicates over TCP and reports encoder and pulse positions for the X, Y, and Z axes.

The tab supports:

- incremental X/Y/Z movement in millimetres;
- reading the current rig position;
- saving a position in the current GUI session; and
- returning to the saved position.

A positive value moves in the configured positive axis direction and a negative value moves in the opposite direction. Software entry ranges are not a substitute for physical travel-limit checks. Motion is monitored until the controller stops and the target position is stable.

## 3. Excitation Mode tab

This tab defines the waveform shared by the acquisition modes and can preview or transmit it.

| Parameter | Meaning | Typical default |
| --- | --- | --- |
| Shape | Carrier waveform supported by the generator | `SIN` |
| Window | Envelope applied to a finite sine burst | Rectangular/None |
| Frequency | Carrier frequency | 1 MHz |
| Amplitude | Generator output amplitude in Vpp | 1 V |
| Cycles per pulse | Carrier cycles in one burst | 5 |
| Number of pulses | Bursts in one acquisition | 2 |
| PRF | Pulse repetition frequency | 100 Hz |
| Start delay | Delay before pulse generation | 0 s |

For carrier frequency $f_c$, cycle count $N_c$, and pulse repetition frequency $PRF$:

$$T_p = \frac{N_c}{f_c}, \qquad T_r = \frac{1}{PRF}$$

The pulse width must satisfy $T_p < T_r$. The preview shows the requested waveform; it does not independently measure the electrical signal at the generator output. The `Run Timing Test` button runs this test and also checks with hardware limitations for the signal generator and displays the outputs.

## 4. A-Mode tab

A-mode performs a single-position pulse-echo measurement. The rig first applies the requested incremental $\Delta X$, $\Delta Y$, and $\Delta Z$, then the oscilloscope captures one waveform per configured pulse. Multiple echoes can be averaged.

| Parameter | Meaning |
| --- | --- |
| ΔX, ΔY, ΔZ | Incremental movement before acquisition, in mm |
| Speed of Sound | Converts round-trip time to depth; `0` retains a time axis |
| High-pass Cutoff | Removes low-frequency content; entered in kHz |
| Filter Order | Butterworth high-pass filter order |
| Average Echoes | Number of acquired echoes combined in the result |
| Live Preview | Updates the waveform plot during acquisition |
| Dry Run | Uses generated echoes instead of instrument data |

For sound speed $c$ and pulse-echo travel time $t$, depth is

$$d = \frac{ct}{2}$$

The factor of two accounts for propagation to the reflector and back. The filter cut-off must be below the Nyquist frequency $f_s/2$. A-mode runs are saved in numbered `data/a_mode_scan_NNN/` folders and can also be exported from the GUI.

## 5. B-Mode tab

B-mode is a linear pulse-echo scan. The rig moves along the selected scan axis and records an A-scan at each point; the waveforms are stacked into a two-dimensional matrix. The other selected axis represents acoustic depth.

| Parameter | Meaning |
| --- | --- |
| Depth Axis | Axis represented by waveform time/depth |
| Scan Axis | Physical axis moved between acquisitions |
| Scan Length | Total commanded distance in mm; negative values reverse direction |
| Scan Points | Number of spatial acquisitions |
| Speed of Sound | Converts time to pulse-echo depth; `0` displays time |
| Dry Run | Generates synthetic A-scans without hardware acquisition |
| Live Preview | Refreshes the B-mode image as rows are acquired |

For a requested length $L$ and the implementation's point count $N$, the commanded step is calculated as

$$\Delta x = \frac{L}{\max(1,N)}$$

Use the export control to save the resulting B-mode matrix.

## 6. 3D-Mode tab

3D mode performs a **planar** scan over two physical axes and records an A-scan at every grid location. Those two spatial dimensions plus waveform time/depth form the 3D dataset. It does not mechanically scan all three axes.

| Parameter | Meaning |
| --- | --- |
| Scan Axis 1 / 2 | Two distinct physical motion axes |
| Length 1 / 2 | Travel along each selected axis in mm |
| Points 1 / 2 | Acquisition-grid dimensions |
| Depth Axis | Remaining axis, selected automatically |
| Scanning Algorithm | `Raster` returns to the same edge; `Zigzag` alternates row direction |
| Live Preview | Updates the selected representation while scanning |
| Dry Run | Uses synthetic echoes and avoids instrument acquisition |

The post-processing selector provides three views of the same type of planar acquisition:

- **A-Mode:** retains the waveform information across the grid.
- **C-Mode:** applies a time gate to the envelope at every grid point, then maps Max, Mean, RMS, Kurtosis, or Energy.
- **Pressure Field Mode:** maps a selected statistical feature and can optionally apply a high-pass or band-pass Butterworth filter.

Pressure-field metrics include Max, Min, Mode, Median, Mean, RMS, Variance, Kurtosis, Skewness, Entropy, and Energy. Available colour maps include 16-bit grey, viridis, plasma, inferno, magma, cividis, turbo, and jet.

## 7. Signal processing and metrics

1. For sampled voltage $x[n]$, the analytic-signal envelope is

$$e[n] = |x[n] + j\,\mathcal{H}\{x[n]\}|$$

where $\mathcal{H}$ is the Hilbert transform. In pulsed excitations with Number of pulses > 1, the envelope is calculated for the averaged signal.

2. C-mode uses samples for which

$$t_0 \leq t \leq t_0 + T_g$$

with gate start $t_0$ and gate width $T_g$. Common reductions of a gated sequence $u_i$ are:

$$\operatorname{Mean}(u)=\frac{1}{N}\sum_{i=1}^{N}u_i$$

$$\operatorname{RMS}(u)=\sqrt{\frac{1}{N}\sum_{i=1}^{N}u_i^2}$$

$$\operatorname{Energy}(u)=\sum_{i=1}^{N}u_i^2$$


3. Filtering is implemented with Butterworth second-order sections. For a valid digital high-pass filter, $0 < f_h < f_s/2$; for a band-pass filter, $0 < f_l < f_h < f_s/2$. The GUI reports and disables invalid cut-offs rather than applying an undefined filter.

4. Dry-run echoes are synthetic test signals. They are useful for checking acquisition flow, plotting, filtering, and export, but they do not validate hardware timing, acoustic calibration, sensitivity, or absolute pressure.

## 8. Data, previews, and exports

Runtime acquisition data are stored under the repository's `data/` directory. Numbered run folders and matrix names prevent normal scans from replacing previous results. Depending on the mode, saved artefacts include:

- waveform CSV files with time and amplitude columns;
- scan-point manifests and acquisition metadata;
- NumPy `.npy` waveform matrices; and
- exported plots, processed data, and tab logs selected by the user.

Matplotlib toolbars provide plot navigation. Live preview affects display updates, not the underlying scan geometry. Do not treat voltage values as calibrated acoustic pressure unless the complete measurement chain—including hydrophone/probe sensitivity, amplification, attenuation, and frequency response—has been calibrated and applied.

## 9. Library modules and troubleshooting

The main source-level modules are:

| Module | Responsibility |
| --- | --- |
| `api.src.rig_function` | Rig TCP connection, position queries, homing, and motion |
| `api.src.Signal_function` | Continuous, triggered, and burst waveform generation |
| `api.src.Oscilloscope` | VISA connection, LeCroy setup, capture, and CSV writing |
| `api.src.scan_utils` | Scan-step calculation |
| `api.src.dummy_signal_generator` | Synthetic echoes for dry runs |

Common checks:

- If the GUI does not start, install the dependencies and launch it from the repository root.
- If VISA discovery fails, confirm the configured resource string, backend/runtime, cable, and network route.
- If the rig is unreachable, check its host, port, subnet, and whether another client owns the connection.
- If filtering is rejected, keep all cut-offs below half the configured sampling rate.
- If hardware is unavailable, enable Dry Run to test processing and export.

To rebuild the HTML page consumed by the GUI after editing this notebook:

```bash
jupyter nbconvert --to html --output docs.html api/docs/documentation.ipynb
```

Run the command from the repository root and commit both the notebook and regenerated HTML when documentation changes must appear inside the application.